# Simulation Based Inference to Remove Sampling Bias - Household Studies


Household infection model with parameters `alpha`, `beta`, `delta`, `mu_inf_SI`, `mu_inf_SC`, `mu_inf_AI`, `mu_inf_AC`, `mu_inf_AA`, `mu_susc_I`, `mu_susc_C`.

In [ ]:
import os
os.environ['KERAS_BACKEND'] = 'jax'

import pickle
import itertools
from matplotlib import pyplot as plt
from tqdm import tqdm
from joblib import Parallel, delayed

import numpy as np
import pandas as pd

import keras
import bayesflow as bf

from PedCov.stan import get_stan_posterior
from PedCov.simulator import OutbreakSimulator
from PedCov.helper_functions import list_of_dicts_to_dict_of_lists, plot_delay_distribution, plot_incubation_distribution, plot_generation_time_distribution

In [ ]:
job_array_id = int(os.environ.get('SLURM_ARRAY_TASK_ID', 0))
n_procs = int(os.environ.get('SLURM_CPUS_PER_TASK', 1))
batch_size = 64

## Define model and prior

We fix some parameters such that the model becomes identifiable (checked with STAN). We fix the parameters `alpha`, `mu_protect_acq`, `mu_protect_transm`.

In [ ]:
param_names = {  # comment out parameters that should not be estimated
    #'alpha': r'$\alpha$',
    'beta': r'$\beta$', 'delta': r'$\delta$',
    'mu_inf_SI': r'$\mu_\text{infectiousness}^\text{symptomatic Infant}$', 'mu_inf_SC': r'$\mu_\text{infectiousness}^\text{symptomatic Child}$',
    'mu_inf_AI': r'$\mu_\text{infectiousness}^\text{asymptomatic Infant}$', 'mu_inf_AC': r'$\mu_\text{infectiousness}^\text{asymptomatic Child}$', 'mu_inf_AA': r'$\mu_\text{infectiousness}^\text{asymptomatic Adult}$',
    'mu_susc_I': r'$\mu_\text{susceptibility}^\text{Infant}$', 'mu_susc_C': r'$\mu_\text{susceptibility}^\text{Child}$',
    #'mu_protect_acq': r'$\mu_\text{protect}^\text{acq}$', 'mu_protect_transm': r'$\mu_\text{protect}^\text{transm}$'
}

full_name_list = ['alpha', 'beta', 'delta',
                  'mu_inf_SI', 'mu_inf_SC', 'mu_inf_AI', 'mu_inf_AC', 'mu_inf_AA',
                  'mu_susc_I', 'mu_susc_C',
                  'mu_protect_acq', 'mu_protect_transm']

In [ ]:
# define the prior
def meta() -> dict:
    selection_procedure_id = np.random.choice([0, 1, 2])  # 0 for random, 1 for pedcov, 2 for adult
    return dict(
        #variant='alpha',  # alpha, omicron
        #variant_id=0,  # 0 for alpha, 1 for omicron
        selection_procedure=['random', 'pedcov', 'adultcov'][selection_procedure_id],
        selection_procedure_id=selection_procedure_id
    )

def prior() -> dict:
    var = 0.7 # was 1 before
    params = {
        'alpha': np.random.uniform(0, 0.1) if 'alpha' in param_names.keys() else 0.001,
        #'beta': np.random.uniform(0, 3) if 'beta' in param_names.keys() else 0.3,
        #'delta': np.random.uniform(-3, 3) if 'delta' in param_names.keys() else 0.1,
        #'alpha': np.random.gamma(shape=1.0, scale=1/20.0) if 'alpha' in param_names.keys() else 0.001,
        'beta': np.random.gamma(shape=2.0, scale=1/2.0) if 'beta' in param_names.keys() else 0.3,
        'delta': np.random.normal(0.0, 1.0) if 'delta' in param_names.keys() else 0.1,
        'mu_inf_SI': np.random.lognormal(0, var) if 'mu_inf_SI' in param_names.keys() else 1.0,
        'mu_inf_SC': np.random.lognormal(0, var) if 'mu_inf_SC' in param_names.keys() else 1.0,
        'mu_inf_AI': np.random.lognormal(0, var) if 'mu_inf_AI' in param_names.keys() else 1.0,
        'mu_inf_AC': np.random.lognormal(0, var) if 'mu_inf_AC' in param_names.keys() else 1.0,
        'mu_inf_AA': np.random.lognormal(0, var) if 'mu_inf_AA' in param_names.keys() else 1.0,
        'mu_susc_I': np.random.lognormal(0, var) if 'mu_susc_I' in param_names.keys() else 1.0,
        'mu_susc_C': np.random.lognormal(0, var) if 'mu_susc_C' in param_names.keys() else 1.0,
        'mu_protect_acq': np.random.lognormal(0, var) if 'mu_protect_acq' in param_names.keys() else 0.8,
        'mu_protect_transm': np.random.lognormal(0, var) if 'mu_protect_transm' in param_names.keys() else 1.0
    }
    return params


def plot_priors(n_samples=10000):
    # Sample from the prior
    samples = {key: np.zeros(n_samples) for key in param_names}
    for i in range(n_samples):
        p = prior()
        for key in samples:
            samples[key][i] = p[key]

    # Set up subplots
    fig, axes = plt.subplots(2, 5, figsize=(10, 4), tight_layout=True)
    axes = axes.flatten()

    for idx, key in enumerate(samples):
        data = samples[key]
        ax = axes[idx]

        # Plot histogram
        ax.hist(data, bins=50, density=True)
        ax.set_title(key)
        ax.set_ylabel('Density')
        ax.set_xlabel(key)

    # Remove any empty subplots
    for j in range(len(samples), len(axes)):
        fig.delaxes(axes[j])
    plt.show()

# Execute the plot function
plot_priors()
prior()

In [ ]:
simulator = OutbreakSimulator()

simulator_bf = lambda alpha, beta, delta, mu_inf_SI, mu_inf_SC, mu_inf_AI, mu_inf_AC, mu_inf_AA, mu_susc_I, mu_susc_C, mu_protect_acq, mu_protect_transm, selection_procedure, return_df=False: simulator(
    alpha=alpha, beta=beta, delta=delta,
    mu_inf_SI=mu_inf_SI, mu_inf_SC=mu_inf_SC, mu_inf_AI=mu_inf_AI, mu_inf_AC=mu_inf_AC, mu_inf_AA=mu_inf_AA,
    mu_susc_I=mu_susc_I, mu_susc_C=mu_susc_C, mu_protect_acq=mu_protect_acq, mu_protect_transm=mu_protect_transm,
    selection_procedure=selection_procedure, return_df=return_df
)
generative_model = bf.make_simulator([meta, prior, simulator_bf]) #, meta_fn=meta)

In [ ]:
#plot_incubation_distribution(simulator.shapeIncub, simulator.scaleIncub, simulator.shapeIncubAsymp, simulator.scaleIncubAsymp)
#plot_generation_time_distribution(simulator.shape_generation_time, simulator.scale_generation_time)
#plot_delay_distribution(simulator.delayDist)

In [ ]:
%%time
test_params = prior()
print(test_params)
test = simulator(**test_params, return_df=True)
test['sim_data_df'].head()
test['sim_data_df'].infect_status.value_counts()

In [ ]:
# # test the STAN posterior
# list_target = []
# list_estimate = []
# for i in range(10):
#     test_params = prior()
#     test_data = simulator(**test_params, return_df=True)
#     test_stan = get_stan_posterior(test_data['sim_data_df'], param_names, simulator, show_progress=True)
#
#     list_target.append(test_params)
#     list_estimate.append(test_stan)
#
# list_target = list_of_dicts_to_dict_of_lists(list_target)
# list_estimate = list_of_dicts_to_dict_of_lists(list_estimate)

In [ ]:
# fig = bf.diagnostics.recovery(list_estimate, list_target, variable_names=list(param_names.values()))
# fig.savefig('plots/test_stan.png', bbox_inches='tight')
# bf.diagnostics.calibration_ecdf(list_estimate, list_target, variable_names=list(param_names.values()), difference=True);

## Generate training data

In [ ]:
num_training_batches = 500
num_validation_sets = 10
training_data_file = f'models/training_data_pedcov.pickle'
validation_data_file = f'models/valid_data_pedcov.pickle'

In [ ]:
@delayed
def simulate_batch(compute_stan_posterior=False) -> dict:
    out = meta()
    out2 = prior()
    if compute_stan_posterior:
        out3 = simulator_bf(selection_procedure=out['selection_procedure'], return_df=True, **out2)
        stan_posterior = get_stan_posterior(out3['sim_data_df'], param_names, simulator, show_progress=False)
        out3.pop('sim_data_df')  # remove the DataFrame, we do not need it anymore
        for k in stan_posterior.keys():
            out3[f'stan_{k}'] = stan_posterior[k]
    else:
        out3 = simulator_bf(selection_procedure=out['selection_procedure'], **out2)
    out.update(out2)
    out.update(out3)
    for p in full_name_list:
        out[p] = [out[p]]
    return out

In [ ]:
if os.path.exists(validation_data_file):
    # load simulation PedCov
    with open(validation_data_file, 'rb') as f:
        validation_data = pickle.load(f)
    try:
        with open(training_data_file, 'rb') as f:
            training_data = pickle.load(f)
    except FileNotFoundError:
        training_data = None
else:
    training_data = Parallel(n_jobs=n_procs, verbose=1)(simulate_batch() for _ in range(batch_size * num_training_batches))
    training_data = list_of_dicts_to_dict_of_lists(training_data)
    with open(training_data_file, 'wb') as f:
        pickle.dump(training_data, f)

    # stan uses 4 cpus per job, so we can use 1/4 of the available cpus
    validation_data = Parallel(n_jobs=n_procs // 4, verbose=1)(simulate_batch(compute_stan_posterior=True) for _ in range(batch_size * num_validation_sets))
    validation_data = list_of_dicts_to_dict_of_lists(validation_data)
    with open(validation_data_file, 'wb') as f:
        pickle.dump(validation_data, f)
    #exit()

# get all random selection procedure data
validation_data_random = validation_data.copy()
select_id = validation_data_random['selection_procedure'] == 'random'
for k in validation_data_random.keys():
    validation_data_random[k] = validation_data_random[k][select_id]

validation_data_pedcov = validation_data.copy()
select_id = validation_data_pedcov['selection_procedure'] == 'pedcov'
for k in validation_data_pedcov.keys():
    validation_data_pedcov[k] = validation_data_pedcov[k][select_id]

validation_data_adultcov = validation_data.copy()
select_id = validation_data_adultcov['selection_procedure'] == 'adultcov'
for k in validation_data_adultcov.keys():
    validation_data_adultcov[k] = validation_data_adultcov[k][select_id]

print(f"Training data shape: {training_data['sim_data'].shape}")
print(f"Validation data shape: {validation_data['sim_data'].shape}")
print(f"Validation data Random: {validation_data_random['sim_data'].shape[0]}")
print(f"Validation data PedCov: {validation_data_pedcov['sim_data'].shape[0]}")
print(f"Validation data AdultCov: {validation_data_adultcov['sim_data'].shape[0]}")

In [ ]:
for vd in [validation_data_random, validation_data_pedcov, validation_data_adultcov]:
    if len(vd['selection_procedure']) == 0:
        continue  # only for testing
    print(f"Validation data shape: {vd['selection_procedure'][0]}")
    stan_posterior_samples = {p: vd[f'stan_{p}'] for p in param_names.keys()}

    fig = bf.diagnostics.recovery(stan_posterior_samples, vd, variable_names=list(param_names.values()),
                                  add_corr=False, figsize=(12, 5))
    ax = fig.get_axes()
    for i, a in enumerate(ax):
        if i != 7:
            a.set_xlabel("")
    plt.tight_layout()
    #fig.savefig(f'plots/pedcov_recovery_stan_{vd['selection_procedure'][0]}.pdf', bbox_inches='tight')
    plt.show()

    fig = bf.diagnostics.calibration_ecdf(stan_posterior_samples, vd, variable_names=list(param_names.values()),
                                          difference=True, figsize=(12, 5))
    ax = fig.get_axes()
    for i, a in enumerate(ax):
        a.get_legend().remove()
        if i != 7:
            a.set_xlabel("")
    plt.tight_layout()
    #fig.savefig(f'plots/pedcov_ecdf_stan_{vd['selection_procedure'][0]}.pdf', bbox_inches='tight')
    plt.show()

## Neural Posterior Estimation

In [ ]:
adapter = (
    bf.adapters.Adapter()
    .drop('selection_procedure')  # we do not need the selection procedure for the adapter
    .to_array()
    .convert_dtype(from_dtype="float64", to_dtype="float32")

    .constrain([k for k in param_names.keys() if k != 'delta'], lower=0, inclusive='none', method="exp")
    .concatenate(param_names.keys(), into="inference_variables")
    .standardize('inference_variables')

    .expand_dims('selection_procedure_id', axis=-1)
    .rename('selection_procedure_id', to_key="inference_conditions")

    .rename('sim_data', to_key="summary_variables")
)

In [ ]:
from bayesflow.utils.serialization import serializable

@serializable("bayesflow.networks")
class DoubleSummaryNetwork(bf.networks.SummaryNetwork):
    def __init__(self, inner_network, outer_network, name=None, **kwargs):
        super().__init__(**kwargs)
        self.name = 'inner_' + inner_network.name + '_outer_' + outer_network.name if name is None else name
        self.inner_network = inner_network  # operates over elements
        self.outer_network = outer_network  # operates over observations

    def call(self, x, training: bool = False, **kwargs):
        b_size, n_outer_obs, n_inner_obs = keras.ops.shape(x)[:3]

        # Flatten to combine batch and outer observation dimensions
        x_flat = keras.ops.reshape(x, (b_size * n_outer_obs, n_inner_obs, *keras.ops.shape(x)[3:]))

        # Apply the inner network to each element in the outer observation
        inner_output = self.inner_network(x_flat, training=training, **kwargs)

        # Reshape back to (b_size, n_outer_obs, inner_output_dim)
        inner_output = keras.ops.reshape(inner_output, (b_size, n_outer_obs, *keras.ops.shape(inner_output)[1:]))

        # Apply the outer network to the inner outputs
        outer_output = self.outer_network(inner_output, training=training, **kwargs)
        return outer_output


    def get_config(self):
        config = super().get_config()
        config.update({
            "inner_network": self.inner_network,
            "outer_network": self.outer_network
        })
        return config

In [ ]:
epochs = 100
inf_i, in_summary_dim, out_summary_dim = list(itertools.product([0, 1], [4, 8, 16], [4, 8, 16]))[job_array_id]
summary_network = DoubleSummaryNetwork(
         inner_network=bf.networks.TimeSeriesNetwork(summary_dim=in_summary_dim),
         outer_network=bf.networks.DeepSet(summary_dim=out_summary_dim),
         name=f'double_deep_set'
)
inference_network = [bf.networks.CouplingFlow(),
                     #bf.networks.ConsistencyModel(epochs*num_training_batches*batch_size),
                    bf.networks.FlowMatching()][inf_i]

model_name = f'pedcov_double_deep_set_{"coupling_flow" if inf_i == 0 else "flow_matching"}_{in_summary_dim}_{out_summary_dim}.keras'

workflow = bf.BasicWorkflow(
    adapter=adapter,
    summary_network=summary_network,
    inference_network=inference_network
)

model_path = f'models/{model_name}'
print(model_path)
if os.path.exists(model_path):
    workflow.approximator = keras.saving.load_model(filepath=model_path)
else:
    history = workflow.fit_offline(
        data=training_data,
        epochs=epochs,
        batch_size=batch_size,
        validation_data=validation_data,
    )
    #workflow.approximator.save(model_path)

In [ ]:
diagnostics = workflow.plot_default_diagnostics(test_data=validation_data, num_samples=300,
                                                calibration_ecdf_kwargs={'difference': True})

#diagnostics['losses'].savefig(f'plots/pedcov_losses_{model_name}.pdf', bbox_inches='tight')
#diagnostics['recovery'].savefig(f'plots/pedcov_recovery_{model_name}.pdf', bbox_inches='tight')
#diagnostics['calibration_ecdf'].savefig(f'plots/pedcov_ecdf_{model_name}.pdf', bbox_inches='tight')

# Simulation Study on Validation Data

In [ ]:
for vd in [validation_data_random, validation_data_pedcov, validation_data_adultcov]:
    if len(vd['selection_procedure']) == 0:
        continue  # only for testing
    print(f"Validation data shape: {vd['selection_procedure'][0]}")
    posterior_samples = workflow.sample(num_samples=1000, conditions=vd)

    fig = bf.diagnostics.recovery(posterior_samples, vd, variable_names=list(param_names.values()),
                                  add_corr=False, figsize=(12, 5))
    ax = fig.get_axes()
    for i, a in enumerate(ax):
        if i != 7:
            a.set_xlabel("")
    plt.tight_layout()
    #fig.savefig(f'plots/pedcov_recovery_{vd['selection_procedure'][0]}.pdf', bbox_inches='tight')
    plt.show()

    fig = bf.diagnostics.calibration_ecdf(posterior_samples, vd, variable_names=list(param_names.values()),
                                          difference=True, figsize=(12, 5))
    ax = fig.get_axes()
    for i, a in enumerate(ax):
        a.get_legend().remove()
        if i != 7:
            a.set_xlabel("")
    plt.tight_layout()
    #fig.savefig(f'plots/pedcov_ecdf_{vd['selection_procedure'][0]}.pdf', bbox_inches='tight')
    plt.show()

## Simulation Study

All parameters are set to 1 besides one parameter. Inference workflow applied on the full model still. Check if for the changing parameter the true parameter is still recovered.

Here we tell the network the right selection procedure. Results should be not be biased for no selection procedure. 

In [ ]:
# generate simulation study
n_sims_per_param = 50
fixed_params = {
    'alpha': 0.001,
    'beta': 0.2,
    'delta': 1.3,
    'mu_protect_acq': 0.8,
    'mu_protect_transm': 1.0,
}
study_params = ['mu_inf_SC', 'mu_inf_SI', 'mu_inf_AI', 'mu_inf_AC', 'mu_inf_AA', 'mu_susc_C', 'mu_susc_I']
selection_procedures = ['pedcov', 'adultcov', 'random']

sim_study_data = []
sim_study_params = []
sim_study_params_stan = {
    'pedcov': [], 'adultcov': [], 'random': []
}
for study_p in study_params:
    params = fixed_params.copy()
    params.update({k: 1. for k in study_params})  # set all parameters to 1
    for _ in range(n_sims_per_param):
        new_param = prior()
        params[study_p] = new_param[study_p]  # set the i-th parameter to a new value

        # run the simulator
        sim_data = simulator(**params, return_df=True, method=selection_procedures)

        # save the simulation PedCov and parameters
        sim_study_data.append(sim_data['sim_data'])
        sim_study_params.append(params.copy())

        # get the posterior samples with STAN
        sim_data_df = sim_data['sim_data_df']
        for sp in selection_procedures:
            sim_data_df_temp = sim_data_df[sim_data_df['selection_process'] == sp]
            sim_study_stan = get_stan_posterior(sim_data_df_temp)
            sim_study_params_stan[sp].append(sim_study_stan)

sim_study_data = list_of_dicts_to_dict_of_lists(sim_study_data)
sim_study_data.update(list_of_dicts_to_dict_of_lists(sim_study_params))
for sp in selection_procedures:
    sim_study_params_stan[sp] = list_of_dicts_to_dict_of_lists(sim_study_params_stan[sp])

# save the PedCov
#with open(f'models/pedcov_sim_study.pickle', 'wb') as f:
#    pickle.dump(sim_study_data, f)
#with open(f'models/pedcov_sim_study_stan.pickle', 'wb') as f:
#    pickle.dump(sim_study_params_stan, f)

In [ ]:
sim_study_data.keys()

In [ ]:
# perform inference on simulation study
results_sim_study = {}
n_samples_study = 10  # increase for more accurate results, e.g. 1000 samples
keep_log_transform = False

for selection_data, selection_inference in tqdm(product(['pedcov', 'random'], ['pedcov', 'random']), total=4):
    for variant in ['alpha', 'omicron']:
        # estimate median and CI for each parameter
        results_sim_study_vs = {
            'posterior_median': np.ones((len(non_fix_indices_study), n_sims_per_param)) * np.nan,
            'posterior_CI': np.ones((len(non_fix_indices_study), 6, n_sims_per_param)) * np.nan,
            'relative_bias_median': np.ones((len(non_fix_indices_study), n_sims_per_param)) * np.nan,
            'relative_bias_CI': np.ones((len(non_fix_indices_study), 6, n_sims_per_param)) * np.nan,
            'overall_median_error': np.ones((len(non_fix_indices_study), n_sims_per_param)) * np.nan,
            'n_infected_rel': np.ones((len(non_fix_indices_study), n_sims_per_param)) * np.nan
        }
        for i, index_i in enumerate(non_fix_indices_study):
            test = sim_study_dict[f'{variant}_{selection_inference}']['sim_batchable_context'][i]
            # get PedCov for this parameter
            sim_study_i = {
                'sim_data': sim_study_dict[f'{variant}_{selection_data}']['sim_data'][i],
                'prior_draws': sim_study_dict[f'{variant}_{selection_data}']['prior_draws'][i],
                # on purpose a different selection procedure than the PedCov was generated with
                'sim_batchable_context': sim_study_dict[f'{variant}_{selection_inference}']['sim_batchable_context'][i],
                'sim_non_batchable_context': sim_study_dict[f'{variant}_{selection_inference}']['sim_non_batchable_context']
            }
            simulation_study_data_conf = trainer.configurator(sim_study_i)
            # get posterior samples
            posterior_samples_sim_study = (
                trainer.amortizer.sample(simulation_study_data_conf, n_samples=n_samples_study)
            )
            posterior_samples_sim_study = renormalize_params(posterior_samples_sim_study,
                                                             keep_log_transform=keep_log_transform)
            if isinstance(simulation_study_data_conf, list):  # for ensemble
                prior_draws_study = renormalize_params(simulation_study_data_conf[0]['parameters'],
                                                       keep_log_transform=keep_log_transform)
            else:
                prior_draws_study = renormalize_params(simulation_study_data_conf['parameters'],
                                                       keep_log_transform=keep_log_transform)

            # get median and CI for each parameter of posterior samples
            # we only care about the changing parameter, the others are set to 1
            results_sim_study_vs['posterior_median'][i] = np.median(posterior_samples_sim_study[:, :, index_i], axis=1)
            results_sim_study_vs['posterior_CI'][i] = np.quantile(posterior_samples_sim_study[:, :, index_i],
                                                                    [0.005, 0.025, 0.1,
                                                                     0.995, 0.975, 0.9],
                                                                    axis=1)
             # get median and CI for each parameter of relative bias (estimate - true) / true
            relative_bias = (posterior_samples_sim_study[:, :, index_i] - prior_draws_study[:, index_i][:, np.newaxis]) / prior_draws_study[:, index_i][:, np.newaxis]
            results_sim_study_vs['relative_bias_median'][i] = np.median(relative_bias, axis=1)
            results_sim_study_vs['relative_bias_CI'][i] = np.quantile(relative_bias,
                                                                        [0.005, 0.025, 0.1,
                                                                         0.995, 0.975, 0.9],
                                                                        axis=1)
            # get overall median error
            results_sim_study_vs['overall_median_error'][i] = np.sum(
                (np.median(posterior_samples_sim_study, axis=1) - prior_draws_study)**2, axis=1
            )

            # get number of infected people in that dataset
            results_sim_study_vs['n_infected_rel'][i] = percentage_infection_age(sim_study_i['sim_data'],
                                                                                 param_names[index_i])

        results_sim_study[f'{selection_data}_{selection_inference}_{variant}'] = results_sim_study_vs

In [ ]:
# plot the results
prior_transformed = renormalize_params(prior_draws_sim_study, keep_log_transform=keep_log_transform)

for selection_data, selection_inference in product(['pedcov', 'random'], ['pedcov', 'random']):
    fig, ax = plt.subplots(2, len(non_fix_indices_study), sharex=True, sharey=True, figsize=(10, 6), tight_layout=True)
    space = 1. / n_sims_per_param
    ci_level = 0.95
    ci_interval = {0.99: [0, 3], 0.95: [1, 4], 0.8: [2, 5]}[0.95]
    colors = ['#1f78b4', '#b2df8a']

    for i, index_i in enumerate(non_fix_indices_study):
        relative_alpha = results_sim_study[f'{selection_data}_{selection_inference}_alpha']['relative_bias_median'][i]
        relative_omicron = results_sim_study[f'{selection_data}_{selection_inference}_omicron']['relative_bias_median'][i]

        # compute the error of the CI
        alpha_err = np.abs(results_sim_study[f'{selection_data}_{selection_inference}_alpha']['relative_bias_CI'][i][ci_interval] -
                           results_sim_study[f'{selection_data}_{selection_inference}_alpha']['relative_bias_median'][i])
        omicron_err = np.abs(results_sim_study[f'{selection_data}_{selection_inference}_omicron']['relative_bias_CI'][i][ci_interval] -
                             results_sim_study[f'{selection_data}_{selection_inference}_omicron']['relative_bias_median'][i])


        plot_alpha = np.ones(prior_transformed[:, index_i].shape[0], dtype=bool)
        plot_omicron = np.ones(prior_transformed[:, index_i].shape[0], dtype=bool)
        n_infected_rel_a = results_sim_study[f'{selection_data}_{selection_inference}_alpha']['n_infected_rel'][i].copy()
        n_infected_rel_o = results_sim_study[f'{selection_data}_{selection_inference}_omicron']['n_infected_rel'][i].copy()
        #plot_alpha = n_infected_rel_a > np.median(n_infected_rel_a)
        #plot_omicron = n_infected_rel_o > np.median(n_infected_rel_o)
        #plot_alpha =  prior_transformed[:, index_i] > 0.5
        for j in range(relative_alpha.shape[0]):
            handle_alpha = ax[0, i].errorbar(prior_transformed[:, index_i][j],
                                             relative_alpha[j],
                                             yerr=alpha_err[:, j][:, np.newaxis],
                                             fmt='o', color=colors[0], alpha=1 if plot_alpha[j] else 0.1,
                                             label='Alpha')
            handle_omicron = ax[1, i].errorbar(prior_transformed[:, index_i][j],
                                               relative_omicron[j],
                                               yerr=omicron_err[:, j][:, np.newaxis],
                                               fmt='o', color=colors[1], alpha=1 if plot_omicron[j] else 0.1,
                                               label='Omicron')

        handle_alpha_mean = ax[0, i].errorbar(1.5,
                                              np.median(relative_alpha[plot_alpha]),
                                              yerr=np.std(relative_alpha[plot_alpha]),
                                              color='black', marker='x', label=r'Median bias ($\pm$ std)', zorder=4)

        handle_omicron_mean = ax[1, i].errorbar(1.5,
                                                np.median(relative_omicron[plot_omicron]),
                                                yerr=np.std(relative_omicron[plot_omicron]),
                                                color='black', marker='x', label=r'Median bias ($\pm$ std)', zorder=4)

        ax[0, i].axhline(0, color='black', ls='--', zorder=3)
        ax[1, i].axhline(0, color='black', ls='--', zorder=3)

        ax[1, i].set_xlabel(param_names[index_i])
        ax[1, i].set_xlim(-0.5, 3.5)
        ax[1, i].set_xticks(np.arange(4))
        ax[0, 0].set_ylim(-1.5, 8)
    ax[0, 0].set_ylabel(f'Relative Bias\n(Median, {round(ci_level*100)}% CI)')
    ax[1, 0].set_ylabel(f'Relative Bias\n(Median, {round(ci_level*100)}% CI)')
    ax[0, n_params//2-1].set_title(f'Simulation Study - Data: {selection_data}, Inference: {selection_inference}')
    fig.legend(handles=[handle_alpha, handle_omicron, handle_alpha_mean],
                loc='lower center', ncol=3, bbox_to_anchor=(0.5, -0.05))
    #fig.savefig(f'{results_folder}/{amortizer_name}_sim_study_data_'
    #            f'{selection_data}_vs_inference_{selection_inference}.png', bbox_inches='tight')
    plt.show()

In [ ]:
# plot bias in both scenarios
fig, ax = plt.subplots(2, len(non_fix_indices_study), sharex=True, sharey=True, figsize=(10, 6), tight_layout=True)
colors = ['#d95f02', '#7570b3']

for j, (selection_data, selection_inference) in enumerate(product(['pedcov', 'random'], ['pedcov', 'random'])):
    if selection_data != selection_inference:
        # using the wrong condition can be used as sanity check
        continue
    for i, index_i in enumerate(non_fix_indices_study):
        # plot a line around 0
        ax[0, i].axhline(0, color='black', ls='--', zorder=3)
        ax[1, i].axhline(0, color='black', ls='--', zorder=3)

        relative_alpha = results_sim_study[f'{selection_data}_{selection_inference}_alpha']['relative_bias_median'][i]
        relative_omicron = results_sim_study[f'{selection_data}_{selection_inference}_omicron']['relative_bias_median'][i]

        # Define the error as [lower_error, upper_error]
        lower_quantile, upper_quantile = np.quantile(relative_alpha, [0.025, 0.975])
        yerr_alpha = np.array([[np.median(relative_alpha) - lower_quantile],
                               [upper_quantile - np.median(relative_alpha)]])
        lower_quantile, upper_quantile = np.quantile(relative_omicron, [0.025, 0.975])
        yerr_omicron = np.array([[np.median(relative_omicron) - lower_quantile],
                                 [upper_quantile - np.median(relative_omicron)]])


        handle_alpha_mean = ax[0, i].errorbar(
            j,
            np.median(relative_alpha),
            yerr=yerr_alpha,
            color=colors[j // 2], marker='x', label=r'Median Bias', zorder=4
        )
        handle_omicron_mean = ax[1, i].errorbar(
            j,
            np.median(relative_omicron),
            yerr=yerr_omicron,
            color=colors[j // 2], marker='x', label=r'Median Bias', zorder=4
        )
        ax[0, i].set_title(param_names[index_i])
        #ax[1, i].set_xlim(-0.5, 3.5)
        ax[1, i].set_xlim(-1.5, 4.5)
        ax[1, i].set_ylim(-1, 4)
        #ax[1, i].set_xticks([0, 1, 2, 3], labels=['PedCov', 'Random', 'PedCov', 'Random'], rotation=60)
        ax[1, i].set_xticks([0, 3], labels=['PedCov', 'Random'], rotation=60)
        ax[0, 0].set_ylabel('Median Relative Bias \n Alpha')
        ax[1, 0].set_ylabel('Median Relative Bias \n Omicron')

color_patch = [Patch(color=colors[i], label=f'Data generated with {["PedCov", "Random"][i]}') for i in range(2)]
leg = fig.legend(handles=color_patch, bbox_to_anchor=(0.5, -0.05), loc='lower center', ncol=2)
#fig.savefig(f'{results_folder}/{amortizer_name}_sim_study_data_bias.png', bbox_inches='tight')
plt.show()

# Apply trained model to real data

In [ ]:
# specify the PedCov path
data_path_alpha = 'PedCov/Simulator/pedcovid_data_structure_alpha.txt'  # todo: exchange with real PedCov of alpha
data_path_omicron = 'PedCov/Simulator/pedcovid_data_structure_omicron.txt'  # todo: exchange with real PedCov of omicron
os.makedirs(results_folder+'/real PedCov', exist_ok=True)

community_infection = {  # must be one of the predefined values, otherwise networks are unreliable
    'alpha': 0.002,
    'omicron': 0.02
}

In [ ]:
# load the PedCov
dfs = {
    'alpha': pd.read_csv(data_path_alpha, delimiter=' ', index_col=0),
    'omicron': pd.read_csv(data_path_omicron, delimiter=' ', index_col=0)
}
prior_samples = renormalize_params(prior(1000))

real_data = {
    'alpha': None,
    'omicron': None
}

# prepare the PedCov for neural networks
for variant in ['alpha', 'omicron']:
    household_data = normalize_household_data(dfs[variant], minimal_length=9)[np.newaxis]
    real_data[variant] = {
        'sim_data': household_data,
        'sim_non_batchable_context': variant,
        'sim_batchable_context': ['pedcov', community_infection[variant]]
    }
    real_data[variant].update({'configured_data': trainer.configurator(real_data[variant])})

In [ ]:
# get posterior samples
for variant in ['alpha', 'omicron']:
    posterior_samples_real = (
        trainer.amortizer.sample(real_data[variant]['configured_data'], n_samples=prior_samples.shape[0])
    )

    # save to csv
    #pd.DataFrame(real_data[variant]['posterior_samples'], columns=param_names).to_csv(
    #    f'{results_folder}/real PedCov/posterior_samples_{variant}.csv'
    #)


In [ ]:
# plot posterior samples vs prior
for variant in ['alpha', 'omicron']:
    print(f"Variant: {variant}")
    fig = bf.diagnostics.plot_posterior_2d(
        posterior_draws=real_data[variant]['posterior_samples'],
        prior_draws=prior_samples,
        param_names=param_names,
        label_fontsize=22,
        legend_fontsize=24
    )
    ax = fig.get_axes()
    for i, a in enumerate(ax):
        # plot only on the diagonal
        if i == i // len(param_names) * (len(param_names)+1):
            a.axvline(1, color='b')
    #plt.savefig(f'{results_folder}/real PedCov/posterior_{variant}.png', bbox_inches='tight')
    plt.show()

In [ ]:
# plot credible intervals for each parameter and each variant
for variant in ['alpha', 'omicron']:
    ax = sampling_parameter_cis(real_data[variant]['posterior_samples'], alpha=[99, 95, 80],
                                param_names=param_names, title=f"Real Data Posterior CIs - {variant}")
    # add vertical line at 1 for the mu parameters
    ax.vlines(1, ymin=1.75, ymax=8.25, color='grey', linestyle='--')
    #plt.savefig(f'{results_folder}/real PedCov/CIs_{variant}.png', bbox_inches='tight')
    plt.show()